In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

# Using project root instead of the jupyter notebook path
ROOT = Path.cwd().parents[1]    # Finds the projject root, two folders backward
sys.path.insert(0, str(ROOT))   # Placing the root at the top of list
from src.data_prep.fetch_integrate import get_integrated_data

C:\Users\fezzati\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the same integrated data the pipeline uses
table = get_integrated_data(
    kaggle_download_path=str(ROOT / 'data/raw/walmart'),
    fred_download_path=str(ROOT / 'data/raw/fred'),
)
table.head()

d:\Codes\Python\WDF\src\data_prep\fetch_integrate.py:145: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  walmart_df["Date"] = pd.to_datetime(walmart_df["Date"])


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,consumer_sentiment,advanced_retail_sales,personal_consum_expend,personal_saving_rate,producer_price_index,effective_federal_fund,household_dept_srvc_ratio
0,5,2010-01-10,283178.12,0,71.10,2.603,212.226946,6.768,74.4,302325.0,10056.1,5.6,181.9,0.11,14.51494
1,15,2010-01-10,566945.95,0,59.69,2.840,132.756800,8.067,74.4,302325.0,10056.1,5.6,181.9,0.11,14.51494
2,42,2010-01-10,481523.93,0,86.01,3.001,126.234600,9.003,74.4,302325.0,10056.1,5.6,181.9,0.11,14.51494
3,33,2010-01-10,224294.39,0,91.45,3.001,126.234600,9.265,74.4,302325.0,10056.1,5.6,181.9,0.11,14.51494
4,36,2010-01-10,422169.47,0,74.66,2.567,210.440443,8.476,74.4,302325.0,10056.1,5.6,181.9,0.11,14.51494


# Group-Wise Statistical Analysis
Here, we statistically examine the weekly sales across different store groups.

1. **In-Group Normality**: The test is executed using Shapiro-Wilk test (```stats.shapiro()```), which evaluates the null hypothesis that the data was originiated from a normal distribution.
2. **Inter-Group Homogeneity**: The Levene’s (```stats.levene()```) test reveals if the variances of groups are equal.

In [ ]:
# Check normality within each store type (p-value threshold = 0.05).
ingroup_norm = True
for store, group in table.groupby(by='Store'):
    stat, p_value = stats.shapiro(group['Weekly_Sales'])
    # print(f'Store {store} Shapiro_Wilk p-value: {p_value:0.4f}')
    if p_value < 0.05:
        ingroup_norm = False
    print(f'Store {store:2d}\'s p-value: {p_value:0.4f}')


# Check for homogeneity of variance across stores (p-value threshold = 0.05)
stat, p_value = stats.levene(*[group['Weekly_Sales'].values for store, group in table.groupby('Store')])
print(f'Store Levene\'s p-value: {p_value:0.4f}')
if p_value < 0.05:
    intergroup_homo = False
else:
    intergroup_homo = True

# Specify the type of analysis
if all([ingroup_norm, intergroup_homo]):
    print('INFO >> In-group distributions are normal and groups are homogeneous. Continue with <ANOVA>, parameteric test.')
else:
    print('INFO >> Joint normality and homogeneity conditions not met. Continue with <Kruskal-Wallis, Mann-Whitney U>, non-parameteric test.')

Store  1's p-value: 0.0000
Store  2's p-value: 0.0000
Store  3's p-value: 0.0000
Store  4's p-value: 0.0000
Store  5's p-value: 0.0000
Store  6's p-value: 0.0000
Store  7's p-value: 0.0000
Store  8's p-value: 0.0000
Store  9's p-value: 0.0000
Store 10's p-value: 0.0000
Store 11's p-value: 0.0000
Store 12's p-value: 0.0000
Store 13's p-value: 0.0000
Store 14's p-value: 0.0000
Store 15's p-value: 0.0000
Store 16's p-value: 0.0000
Store 17's p-value: 0.0000
Store 18's p-value: 0.0000
Store 19's p-value: 0.0000
Store 20's p-value: 0.0000
Store 21's p-value: 0.0000
Store 22's p-value: 0.0000
Store 23's p-value: 0.0000
Store 24's p-value: 0.0000
Store 25's p-value: 0.0000
Store 26's p-value: 0.0000
Store 27's p-value: 0.0000
Store 28's p-value: 0.0000
Store 29's p-value: 0.0000
Store 30's p-value: 0.0010
Store 31's p-value: 0.0000
Store 32's p-value: 0.0000
Store 33's p-value: 0.0253
Store 34's p-value: 0.0000
Store 35's p-value: 0.0000
Store 36's p-value: 0.0000
Store 37's p-value: 0.0018
S

In [8]:
# Check if at least one store differs significantly in overal sales. It checks whether the medians are equall.
h_stat, p_value = stats.kruskal(*[group['Weekly_Sales'] for _ , group in table.groupby('Store')])
if p_value < 0.05:
    print(f'INFO >> At least one store has a significant difference in sales with others.')
    print(f'INFO >> "Store" should remain as a feature, in dummy format.')

INFO >> At least one store has a significant difference in sales with others.
INFO >> "Store" should remain as a feature, in dummy format.


# Feature Correlation and Statistical Significance
Here, we examine which store-level and macroeconomic features are associated with weekly sales.

1. **Per-Store Normalization**: Weekly sales are standardized within each store (Z-score: mean = 0, std = 1). Since stores differ significantly in sales level (see Kruskal-Wallis above), this removes the store-size effect so correlations reflect week-to-week movement rather than differences between big and small stores.
2. **Spearman Rank Correlation**: The test (```stats.spearmanr()```) measures the strength and direction of the monotonic relationship between normalized sales and each feature. It is chosen over Pearson because it does not assume normality, which the tests above rejected.
3. **Statistical Significance**: A feature is considered significant if its p-value is below $\alpha = 0.05$. Non-significant features are listed as candidates for removal.

**Note**: Spearman only captures monotonic relationships. A non-significant feature may still help tree-based models through non-linear effects or interactions, so the removal candidates should be confirmed by retraining the models without them and comparing test error.

In [9]:
# Normalize sales per store (mean=0, std=1)
table['Sales_Normalized'] = table.groupby('Store')['Weekly_Sales'].transform(lambda x: (x - x.mean()) / x.std())

# Specify the columns to evaluate
features_to_evaluate = ['Fuel_Price', 'household_dept_srvc_ratio',
                'Temperature', 'Unemployment', 
                'effective_federal_fund', 'personal_consum_expend', 
                'producer_price_index', 'personal_saving_rate', 
                'advanced_retail_sales', 'consumer_sentiment',
                'Holiday_Flag'
                ]

# Run the correlation analysis for each pair of sales and numeric features
results = []
alpha = 0.05
for feature in features_to_evaluate:
    r_stat, p_stat = stats.spearmanr(table['Sales_Normalized'], table[feature])
    results.append({
        'feature': feature,
        'corr': r_stat,
        'p_value': p_stat,
        'is_significant': p_stat < alpha
    })
corr_table = pd.DataFrame(results).sort_values(by='p_value', ascending=True)
print(corr_table)
# Determine what features to remove
non_significant_features = corr_table[corr_table['is_significant'] == False]['feature'].to_list()
print(f'INFO >> Correlated but non-significant numeric features are: {non_significant_features}')
table.drop('Sales_Normalized', axis=1, inplace=True)


                      feature      corr       p_value  is_significant
1   household_dept_srvc_ratio -0.098214  2.877412e-15            True
5      personal_consum_expend  0.095171  2.004013e-14            True
10               Holiday_Flag  0.089388  6.791568e-13            True
8       advanced_retail_sales  0.088979  8.640222e-13            True
7        personal_saving_rate  0.077459  4.916476e-10            True
0                  Fuel_Price  0.044812  3.233423e-04            True
3                Unemployment -0.042627  6.254062e-04            True
9          consumer_sentiment  0.040292  1.225611e-03            True
4      effective_federal_fund -0.022795  6.747838e-02           False
6        producer_price_index  0.010428  4.029592e-01           False
2                 Temperature  0.004391  7.247224e-01           False
INFO >> Correlated but non-significant numeric features are: ['effective_federal_fund', 'producer_price_index', 'Temperature']
